# Telco customer churn — exploratory analysis

IBM Telco churn dataset: profile target balance, numeric distributions, key categorical splits, linear associations among numeric fields, and missing values (`TotalCharges` has whitespace placeholders parsed as missing).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path("..").resolve()
RAW = ROOT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
FIG = ROOT / "reports/figures"
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.shape

## Churn rate

Class counts for `Churn` (Yes / No).


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="Churn", palette="pastel", ax=ax)
ax.set_title("Churn class counts")
ax.set_xlabel("Churn")
fig.tight_layout()
fig.savefig(FIG / "churn_counts.png", dpi=150, bbox_inches="tight")
plt.show()

## Numeric distributions

Histograms with KDE overlays for tenure, monthly charges, and total charges (total charges uses coerced numeric values; rows with missing total charges are omitted from this density view).


In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, numeric_cols):
    plot_df = df.dropna(subset=[col]) if col == "TotalCharges" else df
    sns.histplot(plot_df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
fig.suptitle("Numeric distributions")
fig.tight_layout()
fig.savefig(FIG / "numeric_histograms_kde.png", dpi=150, bbox_inches="tight")
plt.show()

## Contract, internet service, and payment method

Frequency bar charts for three high-signal categorical fields.


In [ ]:
cat_cols = ["Contract", "InternetService", "PaymentMethod"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, palette="muted", ax=ax)
    ax.set_title(col)
fig.suptitle("Categorical frequencies")
fig.tight_layout()
fig.savefig(FIG / "categorical_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## Numeric correlation heatmap

Pearson correlations among numeric columns (`SeniorCitizen`, tenure, monthly charges, total charges).


In [ ]:
num_for_corr = df.select_dtypes(include=["number"]).copy()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(num_for_corr.corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Numeric correlation matrix")
fig.tight_layout()
fig.savefig(FIG / "numeric_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Missing values

Share of rows missing each column. After coercing `TotalCharges` to numeric, placeholder blanks appear as **11** missing values (0.16% of rows).


In [ ]:
missing_pct = (df.isna().mean().sort_values(ascending=False) * 100).to_frame("missing_pct")
missing_pct = missing_pct[missing_pct["missing_pct"] > 0]
missing_pct.round(3)